In [1]:

import pandas as pd
import numpy as np
import itertools
import statsmodels.api as sm

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [3]:
df = pd.read_excel("dataset_ipm_dummy.xlsx")
df.head()

,rata_lama_sekolah,harapan_lama_sekolah,angka_melek_huruf,rasio_guru_siswa,umur_harapan_hidup,jumlah_faskes,prevalensi_stunting,akses_air_bersih,pengeluaran_perkapita,tingkat_pengangguran,...,pdrb_perkapita,akses_internet,jalan_layak,listrik,indeks_ketimpangan,tingkat_kriminalitas,partisipasi_sosial,ipm,ipm_tinggi,kategori_ipm
0,8.295735,12.546804,85.389749,18.823675,70.616765,3.091756,26.819759,61.412661,5872.868329,8.538709,...,15364.857453,63.811977,60.788121,73.404259,0.379644,60.601124,68.968186,60.736470,0,1
1,7.296400,11.689973,85.381984,21.651405,69.060908,3.600682,21.814270,66.138308,9111.105677,9.053489,...,18542.230822,39.016322,41.057578,55.977533,0.323574,39.388511,59.087432,60.979135,0,1
2,7.807041,12.696608,90.609851,18.058659,69.457696,3.903452,21.024260,69.190472,9692.806205,6.271974,...,20193.562984,64.021497,73.634519,75.044211,0.355166,46.033510,58.131118,68.539719,0,2
3,8.504167,13.740349,94.053056,16.154748,68.593890,4.110582,17.144444,69.723055,9499.127858,5.126285,...,21164.034432,69.840342,71.270386,84.034766,0.336339,40.528529,57.347583,70.064053,1,2
4,7.068128,11.785391,85.821328,19.985563,63.441522,0.399989,29.975474,51.243663,8787.696362,6.598727,...,17887.950481,58.381400,63.089739,71.455986,0.300938,46.207714,56.148507,56.877216,0,1


In [4]:
df.isnull().sum()

rata_lama_sekolah        0
harapan_lama_sekolah     0
angka_melek_huruf        0
rasio_guru_siswa         0
umur_harapan_hidup       0
jumlah_faskes            0
prevalensi_stunting      0
akses_air_bersih         0
pengeluaran_perkapita    0
tingkat_pengangguran     0
persentase_kemiskinan    0
pdrb_perkapita           0
akses_internet           0
jalan_layak              0
listrik                  0
indeks_ketimpangan       0
tingkat_kriminalitas     0
partisipasi_sosial       0
ipm                      0
ipm_tinggi               0
kategori_ipm             0
dtype: int64

In [5]:
X = df.drop(columns=['ipm', 'ipm_tinggi', 'kategori_ipm'])
y = df['ipm']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
import itertools
import statsmodels.api as sm
import pandas as pd

def best_subset(X, y):
    best_adj_r2 = -1
    best_model = None
    best_vars = None

    for k in range(1, len(X.columns)+1):
        for combo in itertools.combinations(X.columns, k):
            X_model = sm.add_constant(X[list(combo)])
            model = sm.OLS(y, X_model).fit()

            if model.rsquared_adj > best_adj_r2:
                best_adj_r2 = model.rsquared_adj
                best_model = model
                best_vars = combo

    print("Best Variables:", best_vars)
    print("Best Adjusted R2:", best_adj_r2)
    
    return best_model

In [ ]:
model_bestsubset = best_subset(X_train, y_train)
print(model_bestsubset.summary())

In [ ]:
def forward_selection(X, y):
    remaining = list(X.columns)
    selected = []
    current_score = 0

    while remaining:
        scores = []

        for candidate in remaining:
            X_model = sm.add_constant(X[selected + [candidate]])
            model = sm.OLS(y, X_model).fit()
            scores.append((model.rsquared_adj, candidate))

        scores.sort()
        best_new_score, best_candidate = scores.pop()

        if best_new_score > current_score:
            remaining.remove(best_candidate)
            selected.append(best_candidate)
            current_score = best_new_score
        else:
            break

    return selected

selected_vars = forward_selection(X_train, y_train)
print("Selected Variables:", selected_vars)

In [ ]:
X_final = sm.add_constant(X_train[selected_vars])
model_final = sm.OLS(y_train, X_final).fit()

print(model_final.summary())

In [ ]:
X_test_final = sm.add_constant(X_test[selected_vars])
y_pred = model_final.predict(X_test_final)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print("MSE:", mse)
print("RMSE:", rmse)

# Check Multikolinearitas

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_temp = sm.add_constant(X)
vif = pd.DataFrame()
vif["Variable"] = X_temp.columns
vif["VIF"] = [variance_inflation_factor(X_temp.values, i) 
              for i in range(X_temp.shape[1])]

print(vif)

# Visualiasai Hasil Model

In [ ]:
plt.figure(figsize=(6,6))
sns.scatterplot(x=y_test, y=y_pred)
plt.xlabel("Actual IPM")
plt.ylabel("Predicted IPM")
plt.title("Actual vs Predicted")
plt.show()

# Plot Residual

In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(6,4))
sns.histplot(residuals, kde=True)
plt.title("Distribusi Residual")
plt.show()